# FableForge-14B: Flagship Coding Agent Training

Train on Colab T4 with Unsloth. 4 stages: Behavior Shaping (lr=2e-4), Skill Distillation (lr=1e-4), Error Recovery (lr=5e-5), DPO Alignment (lr=1e-5).

**Model:** Qwen2.5-14B-Instruct (4-bit QLoRA) | **VRAM:** ~13GB peak | **Time:** ~20h

In [ ]:
# Cell 1: Install
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'unsloth[colab-new]@git+https://github.com/unslothai/unsloth.git'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-deps', 'trl>=0.12.0'])
extras = ['transformers>=4.46.0', 'datasets>=3.0.0', 'accelerate>=1.1.0', 'peft>=0.13.0', 'bitsandbytes>=0.44.0', 'torch>=2.1.0', 'huggingface_hub>=0.24.0', 'tqdm', 'sentencepiece', 'protobuf']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', *extras])
import unsloth; print(f'Unsloth {unsloth.__version__}')
import trl; print(f'TRL {trl.__version__}')


In [ ]:
# Cell 2: GPU + Drive
import torch, os
from pathlib import Path
if not torch.cuda.is_available(): raise RuntimeError('No GPU!')
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
print(f'GPU: {gpu_name}, {gpu_mem:.1f} GB')
DRIVE_MOUNTED = False
try:
    from google.colab import drive; drive.mount('/content/drive'); DRIVE_MOUNTED = True
except Exception: print('Drive not mounted')
OUTPUT_DIR = Path('/content/drive/MyDrive/fableforge-14b') if DRIVE_MOUNTED else Path('/content/fableforge-14b')
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
for d in [OUTPUT_DIR, CHECKPOINT_DIR]: d.mkdir(parents=True, exist_ok=True)
def print_vram(stage=''):
    a=torch.cuda.memory_allocated(0)/1024**3; r=torch.cuda.memory_reserved(0)/1024**3
    f=(torch.cuda.get_device_properties(0).total_mem/1024**3)-r
    print(f'VRAM{(" ("+stage+")" if stage else "")}: {a:.2f} GB alloc, {r:.2f} GB reserved, {f:.2f} GB free')


In [ ]:
# Cell 3: Config
HF_USERNAME = 'your-username'  # CHANGE
HF_REPO_ID = f'{HF_USERNAME}/FableForge-14B'
MODEL_CONFIG = {'model_name': 'unsloth/Qwen2.5-14B-Instruct', 'max_seq_length': 4096, 'lora_rank': 64, 'lora_alpha': 128, 'lora_dropout': 0, 'load_in_4bit': True, 'dtype': None}
TRAINING_DEFAULTS = {'per_device_train_batch_size': 1, 'gradient_accumulation_steps': 16, 'warmup_steps': 50, 'logging_steps': 10, 'save_steps': 200, 'save_total_limit': 3, 'optim': 'adamw_8bit', 'weight_decay': 0.01, 'lr_scheduler_type': 'cosine', 'seed': 42, 'fp16': not torch.cuda.is_bf16_supported(), 'bf16': torch.cuda.is_bf16_supported()}
STAGES = {'stage1_behavior_shaping': {'learning_rate': 2e-4, 'num_train_epochs': 1}, 'stage2_skill_distillation': {'learning_rate': 1e-4, 'num_train_epochs': 1}, 'stage3_error_recovery': {'learning_rate': 5e-5, 'num_train_epochs': 1}, 'stage4_dpo_alignment': {'learning_rate': 1e-5, 'num_train_epochs': 1}}
for k,v in MODEL_CONFIG.items(): print(f'  {k}: {v}')
print_vram('config')


In [ ]:
# Cell 4: Load model
from unsloth import FastLanguageModel
import gc
torch.cuda.empty_cache(); gc.collect()
model, tokenizer = FastLanguageModel.from_pretrained(model_name=MODEL_CONFIG['model_name'], max_seq_length=MODEL_CONFIG['max_seq_length'], load_in_4bit=MODEL_CONFIG['load_in_4bit'], dtype=MODEL_CONFIG['dtype'], trust_remote_code=True)
print_vram('after model load')
model = FastLanguageModel.get_peft_model(model, r=MODEL_CONFIG['lora_rank'], lora_alpha=MODEL_CONFIG['lora_alpha'], lora_dropout=MODEL_CONFIG['lora_dropout'], bias='none', use_gradient_checkpointing='unsloth', random_state=42, use_rslora=False, loftq_config=None, target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'])
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad); total=sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,}/{total:,} ({100*trainable/total:.2f}%)')
print_vram('after LoRA')


In [ ]:
# Cell 5: Data (synthetic fallback)
from datasets import Dataset
from tqdm.notebook import tqdm
import random, json
random.seed(42)
print('Preparing datasets... (using synthetic data; replace with Fable-5 for production)')

def mk_behavior():
    return {'conversations': [{'role':'system','content':'You are FableForge, expert coding agent.'},{'role':'user','content':'Fix the bug: '+random.choice(['off-by-one','None check','type error'])},{'role':'assistant','content':'I will analyze systematically and fix the bug.\n```python\ndef safe_func(x):\n    if x is None: return []\n    return [i for i in x if i is not None]\n```'}]}
def mk_skill():
    return {'conversations': [{'role':'system','content':'You are FableForge. Use tools to explore code.'},{'role':'user','content':random.choice(['Find bug','Debug error','Refactor code'])},{'role':'assistant','content':'Approaching systematically...\n1. Read source files\n2. Identify changes\n3. Make precise edits\n4. Verify'}]}
def mk_error():
    errs=[('ModuleNotFoundError','pip install {module}','Install missing module'),('TypeError: NoneType','Add None check','Check for None'),('IndexError: out of range','Add bounds check','Validate index')]
    e,f,d=random.choice(errs)
    return {'conversations': [{'role':'system','content':'You are FableForge. Diagnose errors.'},{'role':'user','content':f'Error: {e}. Fix?'},{'role':'assistant','content':f'Diagnosis: {d}.\nFix: {f}'}]}
def mk_dpo():
    return {'prompt':'<|im_start|>system\nYou are FableForge.<|im_end|>\n<|im_start|>user\nFix bug<|im_end|>','chosen':'<|im_start|>assistant\nSystematic approach: search, diagnose, fix, verify<|im_end|>','rejected':'<|im_start|>assistant\nJust add try/except<|im_end|>'}
N=2000
behavior_ds=Dataset.from_list([mk_behavior() for _ in tqdm(range(N),desc='behavior')])
skill_ds=Dataset.from_list([mk_skill() for _ in tqdm(range(N),desc='skill')])
error_ds=Dataset.from_list([mk_error() for _ in tqdm(range(N),desc='error')])
dpo_ds=Dataset.from_list([mk_dpo() for _ in tqdm(range(N),desc='dpo')])

def fmt(examples):
    return {'text': [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False) for c in examples['conversations']]}
behavior_formatted=behavior_ds.map(fmt,batched=True); skill_formatted=skill_ds.map(fmt,batched=True); error_formatted=error_ds.map(fmt,batched=True)
print(f'Sizes: behavior={len(behavior_ds):,} skill={len(skill_ds):,} error={len(error_ds):,} dpo={len(dpo_ds):,}')
print_vram('data ready')


In [ ]:
# Cell 6: Stage 1 - Behavior Shaping (~6h T4)
from trl import SFTTrainer
from transformers import TrainingArguments
torch.cuda.empty_cache(); gc.collect()
print_vram('before S1')
s1=STAGES['stage1_behavior_shaping']
s1_dir=str(CHECKPOINT_DIR/'stage1_behavior_shaping')
s1_args=TrainingArguments(output_dir=s1_dir, learning_rate=s1['learning_rate'], num_train_epochs=s1['num_train_epochs'], per_device_train_batch_size=1, gradient_accumulation_steps=16, warmup_steps=50, logging_steps=10, save_steps=200, save_total_limit=3, optim='adamw_8bit', weight_decay=0.01, lr_scheduler_type='cosine', seed=42, fp16=TRAINING_DEFAULTS['fp16'], bf16=TRAINING_DEFAULTS['bf16'], report_to='tensorboard')
s1_trainer=SFTTrainer(model=model,tokenizer=tokenizer,train_dataset=behavior_formatted,dataset_text_field='text',max_seq_length=MODEL_CONFIG['max_seq_length'],args=s1_args)
try:
    r=s1_trainer.train(); print(f'S1 done! Loss: {r.training_loss:.4f}')
except RuntimeError as e:
    if 'out of memory' in str(e).lower():
        torch.cuda.empty_cache();gc.collect();s1_trainer.args.gradient_accumulation_steps=32;r=s1_trainer.train();print(f'S1 done (recovery)! Loss: {r.training_loss:.4f}')
    else: raise
model.save_pretrained(s1_dir);tokenizer.save_pretrained(s1_dir)
print_vram('after S1')


In [ ]:
# Cell 7: Stage 2 - Skill Distillation (~5h)
torch.cuda.empty_cache();gc.collect()
s2=STAGES['stage2_skill_distillation']; s2_dir=str(CHECKPOINT_DIR/'stage2_skill_distillation')
s2_args=TrainingArguments(output_dir=s2_dir,learning_rate=s2['learning_rate'],num_train_epochs=s2['num_train_epochs'],per_device_train_batch_size=1,gradient_accumulation_steps=16,warmup_steps=50,logging_steps=10,save_steps=200,save_total_limit=3,optim='adamw_8bit',weight_decay=0.01,lr_scheduler_type='cosine',seed=42,fp16=TRAINING_DEFAULTS['fp16'],bf16=TRAINING_DEFAULTS['bf16'],report_to='tensorboard')
s2_trainer=SFTTrainer(model=model,tokenizer=tokenizer,train_dataset=skill_formatted,dataset_text_field='text',max_seq_length=MODEL_CONFIG['max_seq_length'],args=s2_args)
try: r=s2_trainer.train(); print(f'S2 done! Loss: {r.training_loss:.4f}')
except RuntimeError as e:
    if 'out of memory' in str(e).lower(): torch.cuda.empty_cache();gc.collect();s2_trainer.args.gradient_accumulation_steps=32;r=s2_trainer.train();print(f'S2 done (recovery)! Loss: {r.training_loss:.4f}')
    else: raise
model.save_pretrained(s2_dir);tokenizer.save_pretrained(s2_dir)
print_vram('after S2')


In [ ]:
# Cell 8: Stage 3 - Error Recovery (~5h)
torch.cuda.empty_cache();gc.collect()
s3=STAGES['stage3_error_recovery']; s3_dir=str(CHECKPOINT_DIR/'stage3_error_recovery')
s3_args=TrainingArguments(output_dir=s3_dir,learning_rate=s3['learning_rate'],num_train_epochs=s3['num_train_epochs'],per_device_train_batch_size=1,gradient_accumulation_steps=16,warmup_steps=50,logging_steps=10,save_steps=200,save_total_limit=3,optim='adamw_8bit',weight_decay=0.01,lr_scheduler_type='cosine',seed=42,fp16=TRAINING_DEFAULTS['fp16'],bf16=TRAINING_DEFAULTS['bf16'],report_to='tensorboard')
s3_trainer=SFTTrainer(model=model,tokenizer=tokenizer,train_dataset=error_formatted,dataset_text_field='text',max_seq_length=MODEL_CONFIG['max_seq_length'],args=s3_args)
try: r=s3_trainer.train(); print(f'S3 done! Loss: {r.training_loss:.4f}')
except RuntimeError as e:
    if 'out of memory' in str(e).lower(): torch.cuda.empty_cache();gc.collect();s3_trainer.args.gradient_accumulation_steps=32;r=s3_trainer.train();print(f'S3 done (recovery)! Loss: {r.training_loss:.4f}')
    else: raise
model.save_pretrained(s3_dir);tokenizer.save_pretrained(s3_dir)
print_vram('after S3')


In [ ]:
# Cell 9: Stage 4 - DPO Alignment (~4h)
from trl import DPOConfig, DPOTrainer
torch.cuda.empty_cache();gc.collect()
print_vram('before S4')
s4=STAGES['stage4_dpo_alignment']
dpo_fmt=dpo_ds.map(lambda x:{'prompt':x['prompt'],'chosen':x['chosen'],'rejected':x['rejected']})
s4_dir=str(CHECKPOINT_DIR/'stage4_dpo_alignment')
dpo_cfg=DPOConfig(output_dir=s4_dir,learning_rate=s4['learning_rate'],num_train_epochs=s4['num_train_epochs'],per_device_train_batch_size=1,gradient_accumulation_steps=16,warmup_steps=25,logging_steps=10,save_steps=200,save_total_limit=3,optim='adamw_8bit',weight_decay=0.01,lr_scheduler_type='cosine',seed=42,fp16=not torch.cuda.is_bf16_supported(),bf16=torch.cuda.is_bf16_supported(),max_length=MODEL_CONFIG['max_seq_length'],max_prompt_length=2048,beta=0.1,loss_type='sigmoid',report_to='tensorboard')
dpo_trainer=DPOTrainer(model=model,ref_model=None,args=dpo_cfg,train_dataset=dpo_fmt,processing_class=tokenizer)
try: r=dpo_trainer.train(); print(f'S4 done! Loss: {r.training_loss:.4f}')
except RuntimeError as e:
    if 'out of memory' in str(e).lower(): torch.cuda.empty_cache();gc.collect();dpo_trainer.args.gradient_accumulation_steps=32;dpo_trainer.args.max_length=2048;r=dpo_trainer.train();print(f'S4 done (recovery)! Loss: {r.training_loss:.4f}')
    else: raise
model.save_pretrained(s4_dir);tokenizer.save_pretrained(s4_dir)
print('All 4 stages complete!')
print_vram('after S4')


In [ ]:
# Cell 10: Evaluate
FastLanguageModel.for_inference(model)
TASKS=[{'name':'Bug Fix','prompt':'My sort function returns same list. def sort_list(items):\n    items.sort()\n    return items','check':lambda r:'sort' in r.lower() or 'sorted' in r.lower()},{'name':'Tool Selection','prompt':'Find Python files with deprecated APIs in src/','check':lambda r:'grep' in r.lower() or 'glob' in r.lower()},{'name':'Error Recovery','prompt':'ModuleNotFoundError: No module named flask. Fix?','check':lambda r:'install' in r.lower() or 'pip' in r.lower()},{'name':'Multi-step','prompt':'Refactor monolithic API into microservices step by step','check':lambda r:'step' in r.lower() or 'first' in r.lower()}]
results=[]
for t in TASKS:
    msgs=[{'role':'system','content':'You are FableForge, expert coding agent.'},{'role':'user','content':t['prompt']}]
    inp=tokenizer.apply_chat_template(msgs,tokenize=True,add_generation_prompt=True,return_tensors='pt').to('cuda')
    with torch.no_grad(): out=model.generate(input_ids=inp,max_new_tokens=512,temperature=0.7,top_p=0.9,do_sample=True)
    resp=tokenizer.decode(out[0][inp.shape[1]:],skip_special_tokens=True)
    passed=t['check'](resp)
    results.append({'task':t['name'],'passed':passed})
    print(f"{t['name']}: {'PASS' if passed else 'FAIL'}")
p=sum(1 for r in results if r['passed']); print(f'Benchmark: {p}/{len(results)} ({100*p/len(results):.0f}%)')


In [ ]:
# Cell 11: Export
import shutil
EXPORT_DIR=OUTPUT_DIR/'exports'; EXPORT_DIR.mkdir(parents=True,exist_ok=True)
merged_dir=str(EXPORT_DIR/'fableforge-14b-16bit')
model.save_pretrained_merged(merged_dir,tokenizer,save_method='merged_16bit')
print(f'16-bit: {merged_dir}')
if DRIVE_MOUNTED:
    dp=Path('/content/drive/MyDrive/fableforge-14b/exports/fableforge-14b-16bit'); dp.mkdir(parents=True,exist_ok=True)
    for f in Path(merged_dir).glob('*'): shutil.copy2(f,dp/f.name)
print_vram('after 16-bit')
gguf_dir=str(EXPORT_DIR/'fableforge-14b-gguf')
try: model.save_pretrained_gguf(gguf_dir,tokenizer,quantization_method='q4_k_m'); print(f'GGUF: {gguf_dir}')
except Exception as e:
    try: model.save_pretrained_gguf(gguf_dir,tokenizer,quantization_method='q8_0'); print(f'GGUF (q8_0): {gguf_dir}')
    except: print('GGUF export failed - use llama.cpp later')
lora_dir=str(EXPORT_DIR/'fableforge-14b-lora')
model.save_pretrained(lora_dir);tokenizer.save_pretrained(lora_dir)
print(f'LoRA: {lora_dir}')


In [ ]:
# Cell 12: Upload to HF
from huggingface_hub import HfApi, create_repo
import os
try:
    from google.colab import userdata; HF_TOKEN=userdata.get('HF_TOKEN')
except Exception: HF_TOKEN=os.environ.get('HF_TOKEN',None) or input('HF token: ')
if HF_TOKEN:
    api=HfApi(token=HF_TOKEN); create_repo(repo_id=HF_REPO_ID,exist_ok=True,token=HF_TOKEN)
    api.upload_folder(folder_path=merged_dir,repo_id=HF_REPO_ID,token=HF_TOKEN,commit_message='FableForge-14B 16-bit')
    lr=f'{HF_REPO_ID}-LoRA'; create_repo(repo_id=lr,exist_ok=True,token=HF_TOKEN)
    api.upload_folder(folder_path=lora_dir,repo_id=lr,token=HF_TOKEN,commit_message='FableForge-14B LoRA')
    if os.path.exists(gguf_dir):
        gr=f'{HF_REPO_ID}-GGUF'; create_repo(repo_id=gr,exist_ok=True,token=HF_TOKEN)
        for gf in Path(gguf_dir).glob('*.gguf'): api.upload_file(path_or_fileobj=str(gf),path_in_repo=gf.name,repo_id=gr,token=HF_TOKEN)
    print(f'https://huggingface.co/{HF_REPO_ID}')
else: print('No HF token')
print('FableForge-14B Training Complete!')
print_vram('final')
